# Lab 1: NumPy for Cat and Dog Faces

In this notebook, you will treat **cat and dog face images** as NumPy arrays and build a small hand-crafted feature matrix.

This version focuses on core NumPy image operations and keeps the workflow concrete:

- load an image into a NumPy array
- crop and flip with slicing
- normalize to `[0, 1]`
- convert RGB to grayscale
- compute summaries with `axis=`
- apply a small filter with a kernel and matrix multiplication
- flatten an image into one vector
- engineer features with `np.concatenate(...)` and `np.apply_along_axis(...)`
- stack features into a feature matrix for later machine learning work

**Dataset assumption**

Use the curated cat-and-dog-faces dataset extracted into:

`data/`

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from lab_utils.visualization import (
    plot_feature_vector,
    show_image_gallery,
)

# Safe project root (works in scripts + notebooks)
try:
    PROJECT_ROOT = Path(__file__).resolve().parent
except NameError:
    PROJECT_ROOT = Path.cwd()

DATA_ROOT = PROJECT_ROOT / "data"

LABELS = ("cat", "dog")
LABEL_TO_INDEX = {"cat": 0, "dog": 1}

IMAGE_EXTENSIONS = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp")

SEED = 1234

def label_from_path(path: Path) -> str:
    label = path.parent.name
    if label not in LABEL_TO_INDEX:
        raise ValueError(f"Unexpected label folder: {path}")
    return label


def load_preview_image(path: Path) -> np.ndarray:
    with Image.open(path) as image:
        return np.asarray(image.convert("RGB"))


def list_image_paths(label: str) -> list[Path]:
    label_dir = DATA_ROOT / label
    paths = []
    for pattern in IMAGE_EXTENSIONS:
        paths.extend(label_dir.glob(pattern))
    return sorted(paths)

def shuffled_paths(paths: list[Path], seed_offset: int = 0) -> list[Path]:
    rng = np.random.default_rng(SEED + seed_offset)
    indices = rng.permutation(len(paths))
    return [paths[int(idx)] for idx in indices]

def sample_paths(paths: list[Path], count: int, seed_offset: int) -> list[Path]:
    ordered = shuffled_paths(paths, seed_offset=seed_offset)
    return ordered[: min(count, len(ordered))]


def sample_per_class(paths: list[Path], n_per_class: int, seed_offset: int = 0) -> list[Path]:
    sampled = []
    for label_index, label in enumerate(LABELS):
        label_paths = [path for path in paths if label_from_path(path) == label]
        sampled.extend(sample_paths(label_paths, n_per_class, seed_offset + 50 * label_index))
    return sampled

def split_train_test(paths: list[Path], train_ratio: float = 0.7, seed_offset: int = 0):
    shuffled = shuffled_paths(paths, seed_offset)
    split_idx = int(len(shuffled) * train_ratio)
    return shuffled[:split_idx], shuffled[split_idx:]


# Check dataset exists
expected = [
    DATA_ROOT / "cat",
    DATA_ROOT / "dog",
]
if not all(path.exists() for path in expected):
    raise FileNotFoundError(
        f"Dataset not found at {DATA_ROOT}. Expected 'cat' and 'dog' folders."
    )


# Load all paths
cat_paths = list_image_paths("cat")
dog_paths = list_image_paths("dog")
cat_dog_paths = cat_paths + dog_paths

# Split per class (7:3)
cat_train, cat_test = split_train_test(cat_paths, 0.7, seed_offset=0)
dog_train, dog_test = split_train_test(dog_paths, 0.7, seed_offset=100)

# Combine
train_paths = cat_train + dog_train
test_paths = cat_test + dog_test

print(f"Using dataset from: {DATA_ROOT}")
print(f"Found {len(cat_paths)} cat images")
print(f"Found {len(dog_paths)} dog images")

if len(cat_paths) == 0 or len(dog_paths) == 0:
    raise ValueError("No images found. Check folder paths or file extensions.")

### Visual Helper: Preview the Faces Dataset

Before starting the TODOs, look at a few cat and dog face images from the student-specific subset.


In [ ]:
preview_paths = sample_per_class(cat_dog_paths, n_per_class=3, seed_offset=10)
preview_images = [load_preview_image(path) for path in preview_paths]
preview_titles = [f"{label_from_path(path)}: {path.name}" for path in preview_paths]
show_image_gallery(
    preview_images,
    titles=preview_titles,
    ncols=3,
    figsize=(10, 6),
    suptitle="Cat and dog face preview",
)
plt.show()


## Question 1: Load one image into a NumPy array

Write a function that:

- opens one file from disk
- converts it to RGB
- returns an `H x W x C` NumPy array

This is the starting point for every later NumPy operation in the lab.


In [ ]:
def load_image_np(path: Path) -> np.ndarray:
    """
    Load an image from disk and convert it to a NumPy array.

    Args:
        path: Path to the image file.

    Returns:
        A NumPy array of shape (H, W, 3) with dtype uint8.
    """
    with Image.open(path) as img:
        return np.asarray(img.convert("RGB"))


# ── Quick sanity check ────────────────────────────────────────────────────────
sample_path  = cat_paths[0]
sample_image = load_image_np(sample_path)

print(f"shape : {sample_image.shape}")           # e.g. (480, 640, 3)
print(f"dtype : {sample_image.dtype}")           # uint8
print(f"range : [{sample_image.min()}, {sample_image.max()}]")

show_image_gallery(
    [sample_image],
    titles=[sample_path.name],
    ncols=1,
    figsize=(4, 4),
)
plt.show()

## Question 2: Crop the image with slicing

Implement a centered square crop. Keep the crop size at `48 x 48` for the rest of the lab so the crop is visible and later operations stay consistent.


In [ ]:
def center_crop(image: np.ndarray, crop_size: int = 48) -> np.ndarray:
    """
    Extract a square crop from the center of an image.

    Args:
        image:     Input image array of shape (H, W) or (H, W, C).
        crop_size: Side length of the square crop in pixels.

    Returns:
        Cropped array of shape (crop_size, crop_size[, C]).

    Raises:
        ValueError: If the image is smaller than crop_size in either dimension.
    """
    h, w = image.shape[:2]
    if h < crop_size or w < crop_size:
        raise ValueError(
            f"Image ({h}×{w}) is smaller than crop_size={crop_size}."
        )

    top  = (h - crop_size) // 2
    left = (w - crop_size) // 2
    return image[top : top + crop_size, left : left + crop_size]


# ── Crop & visualise ──────────────────────────────────────────────────────────
cropped_image = center_crop(sample_image, crop_size=48)
print(f"original shape : {sample_image.shape}")
print(f"cropped shape  : {cropped_image.shape}")

show_image_gallery(
    [sample_image, cropped_image],
    titles=["Original", "Center crop (48×48)"],
    ncols=2,
    figsize=(8, 4),
)
plt.show()


## Question 3: Flip the crop horizontally

Mirror the cropped image from left to right using slicing only.


In [ ]:
def flip_horizontal(image: np.ndarray) -> np.ndarray:
    return image[:, ::-1, :]


flipped_image = flip_horizontal(cropped_image)
show_image_gallery(
    [cropped_image, flipped_image],
    titles=["Cropped", "Flipped"],
    ncols=2,
    figsize=(8, 4),
)
plt.show()

## Question 4: Normalize pixels to `[0, 1]`

Convert the cropped RGB image from unsigned integers into `float32` values in the range `[0, 1]`.


In [ ]:
def normalize_01(image: np.ndarray) -> np.ndarray:
    """
    Normalize a uint8 image to the [0, 1] float range.

    Args:
        image: Input array of dtype uint8 with values in [0, 255].

    Returns:
        float32 array with values in [0.0, 1.0], same shape as input.
    """
    return image.astype(np.float32) / 255.0


def show_histograms(uint8_img: np.ndarray, float_img: np.ndarray) -> None:
    """
    Plot pixel-value histograms before and after normalization side by side.

    Args:
        uint8_img: Original image array (uint8, range 0–255).
        float_img: Normalized image array (float32, range 0–1).
    """
    fig, (ax_before, ax_after) = plt.subplots(1, 2, figsize=(10, 4))

    ax_before.hist(uint8_img.ravel(), bins=50, color="steelblue", edgecolor="none")
    ax_before.set_title("Before — uint8  [0, 255]")
    ax_before.set_xlabel("Pixel value")
    ax_before.set_ylabel("Count")

    ax_after.hist(float_img.ravel(), bins=50, color="darkorange", edgecolor="none")
    ax_after.set_title("After — float32  [0, 1]")
    ax_after.set_xlabel("Pixel value")

    fig.tight_layout()
    plt.show()


# ── Normalize ─────────────────────────────────────────────────────────────────
sample_float = normalize_01(cropped_image)

# 1. Side-by-side images (both look identical — only dtype/range differs)
show_image_gallery(
    [cropped_image, sample_float],
    titles=["uint8  [0, 255]", "float32  [0, 1]"],
    ncols=2,
    figsize=(8, 4),
)

# 2. Stats
for label, arr in [("Before", cropped_image), ("After ", sample_float)]:
    print(f"{label}: dtype={arr.dtype}  min={arr.min():.4f}  max={arr.max():.4f}")

# 3. Histogram
show_histograms(cropped_image, sample_float)

## Question 5: Convert RGB to grayscale

Turn the normalized RGB image into a single grayscale array using standard RGB weights 

$GREY = 0.299 \cdot R + 0.587 \cdot G + 0.114 \cdot B$.


In [ ]:
# ITU-R BT.601 luma coefficients (standard for JPEG/SDTV content)
_BT601_WEIGHTS = np.array([0.299, 0.587, 0.114], dtype=np.float32)


def rgb_to_gray(image_float: np.ndarray) -> np.ndarray:
    """
    Convert a float32 RGB image to grayscale using ITU-R BT.601 luma weights.

    Args:
        image_float: float32 array of shape (H, W, 3) with values in [0, 1].

    Returns:
        float32 array of shape (H, W) with values in [0, 1].
    """
    return np.dot(image_float, _BT601_WEIGHTS)


# ── RGB → Grayscale ───────────────────────────────────────────────────────────
sample_gray = rgb_to_gray(sample_float)
print(f"gray shape : {sample_gray.shape}")
print(f"gray dtype : {sample_gray.dtype}")
print(f"gray range : [{sample_gray.min():.4f}, {sample_gray.max():.4f}]")

fig, (ax_rgb, ax_gray) = plt.subplots(1, 2, figsize=(8, 4))

ax_rgb.imshow(sample_float)
ax_rgb.set_title("Normalized RGB")
ax_rgb.axis("off")

ax_gray.imshow(sample_gray, cmap="gray", vmin=0, vmax=1)
ax_gray.set_title("Grayscale (BT.601)")
ax_gray.axis("off")

fig.tight_layout()
plt.show()

## Question 6: Use `axis=` to summarize channels

Compute one mean value per color channel with `axis=(0, 1)`, then choose the brightest channel with `np.argmax(...)`.


In [ ]:
CHANNEL_NAMES = np.array(["Red", "Green", "Blue"])
CHANNEL_COLORS = ["#E74C3C", "#2ECC71", "#3498DB"]


def channel_summary(image_float: np.ndarray) -> tuple[np.ndarray, int]:
    """
    Compute per-channel mean brightness and identify the dominant channel.

    Args:
        image_float: float32 array of shape (H, W, 3) with values in [0, 1].

    Returns:
        means:              float32 array of shape (3,) — mean per channel.
        brightest_channel:  int index (0=R, 1=G, 2=B) of the brightest channel.
    """
    means: np.ndarray = image_float.mean(axis=(0, 1))
    brightest_channel: int = int(np.argmax(means))
    return means, brightest_channel


# ── Per-channel analysis ──────────────────────────────────────────────────────
sample_channel_means, sample_brightest = channel_summary(sample_float)

for name, mean in zip(CHANNEL_NAMES, sample_channel_means):
    marker = " ← brightest" if name == CHANNEL_NAMES[sample_brightest] else ""
    print(f"  {name:<6}: {mean:.4f}{marker}")

# ── Bar chart ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 3))

bars = ax.bar(CHANNEL_NAMES, sample_channel_means, color=CHANNEL_COLORS, width=0.5)
ax.bar_label(bars, fmt="%.3f", padding=4, fontsize=9)

ax.set_title("Average brightness per channel")
ax.set_ylabel("Mean value [0, 1]")
ax.set_ylim(0, min(sample_channel_means.max() * 1.25, 1.0))
ax.grid(axis="y", alpha=0.25, linestyle="--")

fig.tight_layout()
plt.show()

## Question 7: Apply a filter with a kernel and matrix multiplication

Implement a tiny 2D convolution on the grayscale image. At each location:

1. take a `3 x 3` patch
2. flatten the patch and kernel
3. multiply them with `@`

Use the Laplacian kernel from the setup cell.


In [ ]:
# Discrete Laplacian kernel — highlights regions of rapid intensity change
EDGE_KERNEL = np.array(
    [[0,  1,  0],
     [1, -4,  1],
     [0,  1,  0]],
    dtype=np.float32,
)


def convolve2d_matmul(image_gray: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    """
    Convolve a 2-D grayscale image with a kernel using patch-wise dot products.

    Each output pixel is computed as the dot product of the flattened image
    patch and the flattened kernel — equivalent to a valid-mode convolution.

    Args:
        image_gray: float32 array of shape (H, W).
        kernel:     float32 array of shape (Kh, Kw).

    Returns:
        float32 array of shape (H - Kh + 1, W - Kw + 1).
    """
    h, w         = image_gray.shape
    kh, kw       = kernel.shape
    out_h        = h - kh + 1
    out_w        = w - kw + 1
    flat_kernel  = kernel.ravel()           # view, no copy

    filtered = np.empty((out_h, out_w), dtype=np.float32)

    for row in range(out_h):
        for col in range(out_w):
            filtered[row, col] = (
                image_gray[row : row + kh, col : col + kw].ravel() @ flat_kernel
            )

    return filtered


# ── Edge detection ────────────────────────────────────────────────────────────
sample_filtered = convolve2d_matmul(sample_gray, EDGE_KERNEL)
edge_magnitude  = np.abs(sample_filtered)

print(f"input  shape : {sample_gray.shape}")
print(f"output shape : {sample_filtered.shape}")
print(f"response range : [{sample_filtered.min():.4f}, {sample_filtered.max():.4f}]")

fig, (ax_gray, ax_edge) = plt.subplots(1, 2, figsize=(8, 4))

ax_gray.imshow(sample_gray, cmap="gray", vmin=0, vmax=1)
ax_gray.set_title("Grayscale input")
ax_gray.axis("off")

ax_edge.imshow(edge_magnitude, cmap="magma")
ax_edge.set_title("Laplacian edges  |response|")
ax_edge.axis("off")

fig.tight_layout()
plt.show()

## Question 8: Flatten one image into one vector

Take the grayscale crop and turn it into a one-dimensional vector.


In [ ]:
def flatten_image(image: np.ndarray) -> np.ndarray:
    """
    Flatten an image array into a 1-D feature vector (row-major order).

    Args:
        image: Array of any shape (H, W) or (H, W, C).

    Returns:
        1-D float32 array of length H*W or H*W*C.
    """
    return image.ravel()           # view when possible — no copy


# ── Flatten ───────────────────────────────────────────────────────────────────
sample_flat = flatten_image(sample_gray)

print(f"original shape : {sample_gray.shape}  →  flat shape : {sample_flat.shape}")
print(f"total elements : {sample_flat.size:,}")

# ── Plot first 256 values ─────────────────────────────────────────────────────
N_PREVIEW = 256

fig, ax = plt.subplots(figsize=(10, 3))

ax.plot(sample_flat[:N_PREVIEW], color="#4C6FFF", linewidth=0.9, label="pixel value")
ax.axhline(sample_flat[:N_PREVIEW].mean(), color="#E74C3C",
           linewidth=1, linestyle="--", label="mean")

ax.set_title(f"First {N_PREVIEW} grayscale values after flattening")
ax.set_xlabel("Index")
ax.set_ylabel("Value [0, 1]")
ax.set_xlim(0, N_PREVIEW - 1)
ax.set_ylim(0, 1)
ax.legend(fontsize=9)
ax.grid(alpha=0.25, linestyle="--")

fig.tight_layout()
plt.show()

## Question 9: Engineer a feature vector with `concatenate` and `apply`

Build one hand-crafted feature vector that combines:

- RGB means
- RGB standard deviations
- the brightest channel index
- the mean and standard deviation of the filtered response
- one summary from `np.apply_along_axis(...)`

Use `np.concatenate(...)` to join the pieces.


In [ ]:
FEATURE_NAMES: list[str] = [
    "mean_r",           # mean red channel intensity
    "mean_g",           # mean green channel intensity
    "mean_b",           # mean blue channel intensity
    "std_r",            # std red channel intensity
    "std_g",            # std green channel intensity
    "std_b",            # std blue channel intensity
    "brightest_channel",# index of dominant channel (0=R, 1=G, 2=B)
    "edge_mean",        # mean absolute Laplacian response
    "edge_std",         # std of Laplacian response
    "row_std_mean",     # mean per-row std (horizontal texture measure)
]

assert len(FEATURE_NAMES) == 10, "FEATURE_NAMES must stay in sync with extract_features()"


def extract_features(image: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    """
    Compute a 10-dimensional hand-crafted feature vector from an RGB image.

    Pipeline:
        1. Center-crop to 48×48
        2. Normalize to [0, 1]
        3. Per-channel mean & std  (6 features)
        4. Dominant channel index  (1 feature)
        5. Laplacian edge stats    (2 features)
        6. Row-wise std mean       (1 feature)

    Args:
        image:  uint8 RGB array of shape (H, W, 3).
        kernel: float32 convolution kernel, e.g. EDGE_KERNEL.

    Returns:
        float32 array of shape (10,) matching FEATURE_NAMES.
    """
    # ── Pre-processing ────────────────────────────────────────────────────────
    cropped      = center_crop(image, crop_size=48)
    image_float  = normalize_01(cropped)
    gray         = rgb_to_gray(image_float)

    # ── Color features ────────────────────────────────────────────────────────
    channel_means, brightest_channel = channel_summary(image_float)
    channel_stds = image_float.std(axis=(0, 1)).astype(np.float32)

    # ── Texture / edge features ───────────────────────────────────────────────
    filtered         = convolve2d_matmul(gray, kernel)
    row_std_profile  = gray.std(axis=1)                 # (H,) — one std per row

    # ── Assemble ──────────────────────────────────────────────────────────────
    features = np.concatenate([
        channel_means.astype(np.float32),               # [0:3]  mean R/G/B
        channel_stds,                                   # [3:6]  std  R/G/B
        np.array([brightest_channel], dtype=np.float32),# [6]    dominant channel
        np.array([                                      # [7:10] edge + texture
            filtered.mean(),
            filtered.std(),
            row_std_profile.mean(),
        ], dtype=np.float32),
    ])

    assert features.shape == (len(FEATURE_NAMES),), (
        f"Feature length mismatch: got {features.shape[0]}, "
        f"expected {len(FEATURE_NAMES)}"
    )

    return features


# ── Extract & inspect ─────────────────────────────────────────────────────────
sample_features = extract_features(sample_image, EDGE_KERNEL)

print(f"feature shape : {sample_features.shape}")
print(f"{'feature':<20}  {'value':>8}")
print(f"{'─' * 20}  {'─' * 8}")
for name, val in zip(FEATURE_NAMES, sample_features):
    print(f"{name:<20}  {val:>8.4f}")

fig, ax = plot_feature_vector(
    sample_features,
    FEATURE_NAMES,
    title="Sample NumPy feature vector",
)
plt.show()

## Question 10: Build and inspect a feature matrix

Apply your feature function to the small balanced train/test subsets from the face dataset.

Tasks:

1. build one feature matrix for the train images and one for the test images
2. return the matching integer labels
3. print the resulting shapes
4. compute an overall feature mean with `axis=0`
5. visualize the feature matrix and the average feature vector


In [ ]:
def build_feature_matrix(paths: list[Path], kernel: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    x = np.stack([extract_features(load_image_np(path), kernel) for path in paths])
    y = np.array([LABEL_TO_INDEX[label_from_path(path)] for path in paths], dtype=np.int64)
    return x, y


X_train, y_train = build_feature_matrix(train_paths, EDGE_KERNEL)
X_test, y_test = build_feature_matrix(test_paths, EDGE_KERNEL)
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
print("train class counts:", np.bincount(y_train, minlength=2))
print("test class counts:", np.bincount(y_test, minlength=2))

train_feature_mean = X_train.mean(axis=0)
print("overall train feature mean shape:", train_feature_mean.shape)

fig, ax = plt.subplots(figsize=(10, 4))
image = ax.imshow(X_train, aspect="auto", cmap="viridis")
ax.set_title("Train feature matrix")
ax.set_xlabel("Feature index")
ax.set_ylabel("Image index")
ax.set_xticks(range(len(FEATURE_NAMES)))
ax.set_xticklabels(FEATURE_NAMES, rotation=45, ha="right")
fig.colorbar(image, ax=ax, fraction=0.03, pad=0.02)
fig.tight_layout()
plt.show()

fig, ax = plot_feature_vector(train_feature_mean, FEATURE_NAMES, title="Average training feature vector")
plt.show()

## Reflection

Answer these short questions in your own words:

1. Why is it useful to keep the crop size fixed before feature extraction?
2. What does `axis=(0, 1)` mean when you compute channel means on an image?
3. What information does the small edge filter capture that plain RGB means miss?
4. Why does flattening help some operations but also lose spatial structure?
